# Parse market
Fold sorted parsed logs into resumable books. Instrument and book snapshots restore bounded runs.

In [ ]:
source = "fixmessage.market"
target = "market.books"
start = None
end = None
catalog = "rekep"
catalog_properties = {}
snapshot_every = 3_600_000_000_000
max_lateness_ns = 900_000_000_000
max_order_age_ns = 86_400_000_000_000
max_side_alive = 10_000
merge_by = True
commit_row_size = 250_000

In [ ]:
import pyarrow
from pyiceberg.expressions import (
    And,
    GreaterThanOrEqual,
    LessThan,
    LessThanOrEqual,
    NotNull,
)

from rekep.iceberg import IcebergDataset
from rekep.market import Book, BookIterator
from rekep.text import FixMessage
from rekep.times import unix_of

HOUR = 3_600_000_000_000
DAY = 86_400_000_000_000


def _window(lower, upper, column="unix"):
    predicates = []
    if lower is not None:
        predicates.append(GreaterThanOrEqual(column, lower))
    if upper is not None:
        predicates.append(LessThan(column, upper))
    return None if not predicates else predicates[0] if len(predicates) == 1 else And(*predicates)


lower, upper = unix_of(start), unix_of(end, upper=True)
read_lower = None if lower is None else lower - lower % HOUR - HOUR
read_upper = None if upper is None else ((upper + HOUR - 1) // HOUR) * HOUR + max_lateness_ns
logs_table = IcebergDataset(name=source, catalog=catalog, properties=dict(catalog_properties))
book_table = IcebergDataset(
    name=target,
    catalog=catalog,
    properties=dict(catalog_properties),
    field=Book.into_field(),
    commit_row_size=commit_row_size,
    sort_by=("unix", "hash"),
)


def _book_seeds():
    if read_lower is None:
        return []
    recent = And(
        GreaterThanOrEqual("unix", read_lower - DAY),
        LessThanOrEqual("unix", read_lower),
        NotNull("sunix"),
    )
    reader = book_table.read_arrow_reader(
        Book.into_field(), row_filter=recent, order_by=("unix", "hash")
    )
    latest = {}
    for batch in reader:
        for row in batch.to_pylist():
            book = Book.from_dict(row)
            current = latest.get(book.instrument_xhash)
            if current is None or (book.unix, book.version, book.hash) > (
                current.unix,
                current.version,
                current.hash,
            ):
                latest[book.instrument_xhash] = book
    return sorted(latest.values(), key=lambda book: (book.unix, book.instrument_xhash))


snapshots = _book_seeds()
checkpoint = min((book.unix for book in snapshots), default=None)


def _logs():
    reader = logs_table.read_arrow_reader(
        FixMessage.into_field(),
        row_filter=_window(read_lower, read_upper),
        order_by=("unix", "msg_seq_num", "hash"),
    )
    for batch in reader:
        for row in batch.to_pylist():
            yield FixMessage.from_dict(row)


def _flush(held):
    if not held:
        return 0
    table = pyarrow.Table.from_pylist(
        [book.into_dict() for book in held],
        schema=Book.into_field().into_arrow_schema(),
    )
    held.clear()
    return book_table.append_arrow_table(table, merge_by=merge_by)


held = []
read = written = 0
books = BookIterator(
    logs=_logs(),
    snapshots=snapshots,
    snapshot_every=snapshot_every,
    snapshot_until=upper,
    max_order_age_ns=max_order_age_ns,
    max_side_alive=max_side_alive,
)
for book in books:
    if lower is not None and book.unix < lower:
        continue
    if upper is not None and book.unix >= upper:
        continue
    held.append(book)
    read += 1
    if len(held) >= commit_row_size:
        written += _flush(held)
written += _flush(held)
result = {
    "books": read,
    "written": written,
    "checkpoint": checkpoint,
    "read_lower": read_lower,
    "read_upper": read_upper,
    "target": target,
}
result